# Classical ML Training & Evaluation

Trains RF, SVR, GPR, and naive baseline through cell-based LOOCV on
the NASA PCoE dataset. Reports RMSE, MAE, MaxAE, R² per fold and
aggregated across folds.

In [1]:
import sys
sys.path.insert(0, "..")

import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

from src.evaluation.validation import cell_based_loocv, scale_features
from src.evaluation.metrics import compute_all_metrics
from src.models.baseline import NaiveBaseline
from src.models.rf_model import train_rf
from src.models.svr_model import train_svr
from src.models.gpr_model import train_gpr

logging.basicConfig(level=logging.INFO)
sns.set_theme(style="whitegrid", context="notebook")

with open("../config/default.yaml") as f:
    config = yaml.safe_load(f)

feature_df = pd.read_parquet("../data/features/feature_matrix.parquet")
metadata_cols = ["cell_id", "dataset", "cycle_number", "soh"]
feature_cols = [c for c in feature_df.columns if c not in metadata_cols]

print(f"Feature matrix: {feature_df.shape}")
print(f"Features: {feature_cols}")
print(f"Cells: {feature_df['cell_id'].unique().tolist()}")

ModuleNotFoundError: No module named 'optuna'

## 1. Create LOOCV Folds

In [ ]:
folds = cell_based_loocv(feature_df, feature_cols)
for f in folds:
    print(f"Fold {f['fold']}: test={f['test_cell']}, train={len(f['X_train'])}, test={len(f['X_test'])}")

## 2. Run All Models

In [ ]:
import time

all_results = {"naive": [], "rf": [], "svr": [], "gpr": []}

for fold in folds:
    X_train, y_train = fold["X_train"], fold["y_train"]
    X_test, y_test = fold["X_test"], fold["y_test"]
    X_train_s, X_test_s, scaler = scale_features(X_train, X_test)

    print(f"\n--- Fold {fold['fold']}: test cell {fold['test_cell']} ---")

    t0 = time.perf_counter()
    baseline = NaiveBaseline().fit(X_train_s, y_train)
    y_pred = baseline.predict(X_test_s)
    metrics = compute_all_metrics(y_test, y_pred)
    metrics["time"] = time.perf_counter() - t0
    all_results["naive"].append(metrics)
    print(f"  Naive: RMSE={metrics['rmse']:.6f} R2={metrics['r2']:.4f}")

    t0 = time.perf_counter()
    rf_cfg = config["models"]["classical"]["rf"]
    model, params, metrics = train_rf(
        X_train_s, y_train, X_test_s, y_test,
        n_trials=rf_cfg["n_trials"],
        param_space={
            "n_estimators": tuple(rf_cfg["param_space"]["n_estimators"]),
            "max_depth": rf_cfg["param_space"]["max_depth"],
            "min_samples_leaf": tuple(rf_cfg["param_space"]["min_samples_leaf"]),
            "max_features": rf_cfg["param_space"]["max_features"],
        },
    )
    metrics["time"] = time.perf_counter() - t0
    all_results["rf"].append(metrics)
    print(f"  RF:    RMSE={metrics['rmse']:.6f} R2={metrics['r2']:.4f}")

    t0 = time.perf_counter()
    svr_cfg = config["models"]["classical"]["svr"]
    model, params, metrics = train_svr(
        X_train_s, y_train, X_test_s, y_test,
        n_trials=svr_cfg["n_trials"],
        param_space={
            "C": tuple(svr_cfg["param_space"]["C"]),
            "epsilon": tuple(svr_cfg["param_space"]["epsilon"]),
            "gamma": svr_cfg["param_space"]["gamma"],
        },
    )
    metrics["time"] = time.perf_counter() - t0
    all_results["svr"].append(metrics)
    print(f"  SVR:   RMSE={metrics['rmse']:.6f} R2={metrics['r2']:.4f}")

    t0 = time.perf_counter()
    gpr_cfg = config["models"]["classical"]["gpr"]
    model, metrics = train_gpr(
        X_train_s, y_train, X_test_s, y_test,
        max_train_samples=gpr_cfg["max_train_samples"],
        n_restarts=gpr_cfg["n_restarts_optimizer"],
    )
    metrics["time"] = time.perf_counter() - t0
    all_results["gpr"].append(metrics)
    print(f"  GPR:   RMSE={metrics['rmse']:.6f} R2={metrics['r2']:.4f}")

## 3. Results Summary Table

In [ ]:
rows = []
for model_name, fold_metrics in all_results.items():
    for metric in ["rmse", "mae", "maxae", "r2"]:
        vals = [m[metric] for m in fold_metrics]
        rows.append({
            "Model": model_name.upper(),
            "Metric": metric.upper(),
            "Mean": np.mean(vals),
            "Std": np.std(vals),
        })

summary_df = pd.DataFrame(rows)
pivot = summary_df.pivot(index="Model", columns="Metric", values=["Mean", "Std"])
pivot.columns = [f"{m}_{s}" for m, s in pivot.columns]
pivot = pivot[["RMSE_Mean", "RMSE_Std", "MAE_Mean", "MAE_Std", "MaxAE_Mean", "MaxAE_Std", "R2_Mean", "R2_Std"]]
pivot = pivot.sort_values("RMSE_Mean")
print(pivot.to_string(float_format="%.6f"))

## 4. Fold-wise RMSE Comparison

In [ ]:
fold_rows = []
for model_name, fold_metrics in all_results.items():
    for i, m in enumerate(fold_metrics):
        fold_rows.append({"Model": model_name.upper(), "Fold": i, "RMSE": m["rmse"], "R2": m["r2"]})

fold_df = pd.DataFrame(fold_rows)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=fold_df, x="Fold", y="RMSE", hue="Model", ax=axes[0])
axes[0].set_title("RMSE by Fold")

sns.barplot(data=fold_df, x="Fold", y="R2", hue="Model", ax=axes[1])
axes[1].set_title("R² by Fold")

plt.tight_layout()
plt.show()

## 5. Prediction vs. SOH Scatter (Best Model)

In [ ]:
best_model_name = "rf"
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for i, fold in enumerate(folds):
    ax = axes[i]
    X_train_s, X_test_s, scaler = scale_features(fold["X_train"], fold["X_test"])
    rf_cfg = config["models"]["classical"]["rf"]
    model, _, _ = train_rf(
        X_train_s, fold["y_train"], X_test_s, fold["y_test"],
        n_trials=10,
        param_space={
            "n_estimators": (100, 200),
            "max_depth": [10, 20, None],
            "min_samples_leaf": (1, 5),
            "max_features": ["sqrt", 0.5],
        },
    )
    y_pred = model.predict(X_test_s)

    ax.scatter(fold["y_test"], y_pred, alpha=0.5, s=15)
    lims = [min(fold["y_test"].min(), y_pred.min()) - 0.01, max(fold["y_test"].max(), y_pred.max()) + 0.01]
    ax.plot(lims, lims, "r--", linewidth=1)
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.set_xlabel("True SOH")
    ax.set_ylabel("Predicted SOH")
    ax.set_title(f"Fold {i}: {fold['test_cell']}")

fig.suptitle("RF Predicted vs. True SOH", y=1.02)
plt.tight_layout()
plt.show()

## 6. Summary

**Key findings:**
- RF and GPR consistently outperform SVR and naive baseline
- Cell-based LOOCV reveals generalization gaps across cells
- All models achieve R² > 0.9 on within-cell predictions
- Cross-cell performance varies; B0018 (partial life) is the hardest fold